# Task 2: Missing-Data and Degenerate-Value Patterns

For every variable, compute: % null, % zero, % at ceiling (where applicable). Cross-tabulate by basin level (L8 vs. L6) to identify where variables systematically degrade at coarser resolution.

## Key caution: the N-confound

L8 has 190,675 basins; L6 has 16,397 — roughly a 1:12 ratio. L6 distributions will look different from L8 even for variables that don't genuinely degrade, because:
- Smaller N means noisier estimates of rare events (a variable with 1% null at L8 might show 0% or 3% at L6 by chance alone)
- Coarser polygons aggregate what was previously multiple L8 basins — genuinely smoothing out local variation

The goal is to distinguish **genuine scale-degradation** (the variable loses meaningful information at L6) from **statistical artifact** (the difference is within the noise expected from 12× fewer samples).

## Variable naming
Same three-layer naming as Task 1: `basin08/06 column` → `api_key` → `schema_key`. This notebook works at the column level and labels outputs with the api_key. Temperature columns (`tmp_dc_*`) are stored ×10 — divide by 10 for display.

In [ ]:
# Cell 1
import sys
sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from scripts.shared.db_utils import db_connect

print('Imports OK')

In [ ]:
# Cell 2
# Variable definitions
# (basin_col, api_key, scale_factor, units, pct_var, ceiling)
# pct_var: True if variable is a percentage (0-100), so ceiling=100 is meaningful
# ceiling: the theoretical maximum value (None if unbounded)

SCALARS = [
    # Band A — Terrain
    ('ele_mt_smn', 'elev_min',               1.0,  'm',       False, None),
    ('ele_mt_smx', 'elev_max',               1.0,  'm',       False, None),
    ('slp_dg_sav', 'slope_avg',              1.0,  'degrees', False, None),
    ('slp_dg_uav', 'slope_upstream',         1.0,  'degrees', False, None),
    ('sgr_dk_sav', 'stream_gradient',        1.0,  'm/km',    False, None),
    ('kar_pc_sse', 'karst',                  1.0,  '%',       True,  100),
    ('kar_pc_use', 'karst_upstream',         1.0,  '%',       True,  100),
    ('prm_pc_sse', 'permafrost_extent',      1.0,  '%',       True,  100),
    # Band B — Hydrology & soils
    ('dis_m3_pyr', 'discharge_yr',           1.0,  'm³/s',    False, None),
    ('dis_m3_pmn', 'discharge_min',          1.0,  'm³/s',    False, None),
    ('dis_m3_pmx', 'discharge_max',          1.0,  'm³/s',    False, None),
    ('run_mm_syr', 'runoff',                 1.0,  'mm/yr',   False, None),
    ('gwt_cm_sav', 'gw_table_depth',         1.0,  'cm',      False, None),
    ('ria_ha_ssu', 'river_area',             1.0,  'ha',      False, None),
    ('ria_ha_usu', 'river_area_upstream',    1.0,  'ha',      False, None),
    ('wet_pc_sg1', 'wet_pct_grp1',           1.0,  '%',       True,  100),
    ('wet_pc_ug1', 'wet_pct_grp1_upstream',  1.0,  '%',       True,  100),
    ('wet_pc_sg2', 'wet_pct_grp2',           1.0,  '%',       True,  100),
    ('rev_mc_usu', 'reservoir_vol',          1.0,  'km³',     False, None),
    ('cly_pc_sav', 'pct_clay',               1.0,  '%',       True,  100),
    ('slt_pc_sav', 'pct_silt',               1.0,  '%',       True,  100),
    ('snd_pc_sav', 'pct_sand',               1.0,  '%',       True,  100),
    # Band C — Climate
    ('tmp_dc_syr', 'temp_yr',                0.1,  '°C',      False, None),
    ('tmp_dc_uyr', 'temp_yr_upstream',       0.1,  '°C',      False, None),
    ('tmp_dc_smn', 'temp_min',               0.1,  '°C',      False, None),
    ('tmp_dc_smx', 'temp_max',               0.1,  '°C',      False, None),
    ('pre_mm_syr', 'precip_yr',              1.0,  'mm/yr',   False, None),
    ('pre_mm_uyr', 'precip_yr_upstream',     1.0,  'mm/yr',   False, None),
    ('ari_ix_sav', 'aridity',                1.0,  'idx×100', False, None),
    ('ari_ix_uav', 'aridity_upstream',       1.0,  'idx×100', False, None),
    # Band D — Human
    ('crp_pc_sse', 'cropland_extent',        1.0,  '%',       True,  100),
    ('crp_pc_use', 'cropland_extent_upstream',1.0, '%',       True,  100),
    ('ppd_pk_sav', 'pop_density',            1.0,  'pk/km²',  False, None),
    ('hft_ix_s09', 'human_footprint_09',     1.0,  'index',   False, None),
    ('hft_ix_u09', 'human_footprint_09_upstream',1.0,'index', False, None),
    ('gdp_ud_sav', 'gdp_avg',               1.0,  'USD/km²', False, None),
    ('hdi_ix_sav', 'human_dev_idx',          1.0,  'index',   False, None),
    # Coastality
    ('dist_sink',  'dist_sink',              1.0,  'km',      False, None),
]

print(f'{len(SCALARS)} scalar variables defined')

In [ ]:
# Cell 3
# Verify column availability in both tables and pull data
# Using cursor-based fetch to avoid SQLAlchemy/psycopg3 compatibility warning

conn = db_connect()

def get_existing_cols(conn, table):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema = 'public' AND table_name = %s
        """, (table,))
        return {row[0] for row in cur.fetchall()}

def fetch_table(conn, table, cols):
    """Pull specified columns from table, replacing -9999 sentinel with NaN."""
    existing = get_existing_cols(conn, table)
    available = [c for c in cols if c in existing]
    missing   = [c for c in cols if c not in existing]
    if missing:
        print(f'  {table}: columns not found (skipping): {missing}')
    col_sql = ', '.join(f'"{c}"' for c in available)
    with conn.cursor() as cur:
        cur.execute(f'SELECT {col_sql} FROM public.{table}')
        rows = cur.fetchall()
        df = pd.DataFrame(rows, columns=available)
    df = df.replace(-9999, np.nan)
    return df

scalar_cols = [s[0] for s in SCALARS]

print('Loading L8...')
df8 = fetch_table(conn, 'basin08', scalar_cols)
print(f'  L8: {len(df8):,} rows × {len(df8.columns)} columns')

print('Loading L6...')
df6 = fetch_table(conn, 'basin06', scalar_cols)
print(f'  L6: {len(df6):,} rows × {len(df6.columns)} columns')

conn.close()
print('Done.')

## Part 1: Per-variable degenerate-value statistics at L8 and L6

For each variable we compute three things:
- **% null**: data simply absent — no value recorded
- **% zero**: data present but zero — a real value, but one that may mean "absent" for sparse phenomena (karst, permafrost, wetlands)
- **% at ceiling**: for percentage variables (0–100), the fraction of basins at exactly 100 — possible sign of saturation or data truncation

These are computed separately for L8 and L6 so we can compare them.

In [ ]:
# Cell 4
def degenerate_stats(df, basin_col, api_key, ceiling):
    """Compute null/zero/ceiling rates for one variable in one DataFrame."""
    n_total  = len(df)
    if basin_col not in df.columns:
        return dict(api_key=api_key, n_total=n_total,
                    pct_null=None, pct_zero=None, pct_ceiling=None)
    col      = df[basin_col]
    n_null   = col.isna().sum()
    valid    = col.dropna()
    n_zero   = (valid == 0).sum()
    n_ceil   = (valid == ceiling).sum() if ceiling is not None else None
    return dict(
        api_key     = api_key,
        n_total     = n_total,
        pct_null    = round(100 * n_null / n_total, 2),
        pct_zero    = round(100 * n_zero / n_total, 2),
        pct_ceiling = round(100 * n_ceil / n_total, 2) if n_ceil is not None else None,
    )

rows8, rows6 = [], []
for basin_col, api_key, scale, units, pct_var, ceiling in SCALARS:
    rows8.append(degenerate_stats(df8, basin_col, api_key, ceiling))
    rows6.append(degenerate_stats(df6, basin_col, api_key, ceiling))

stats8 = pd.DataFrame(rows8).set_index('api_key')
stats6 = pd.DataFrame(rows6).set_index('api_key')

print('L8 stats computed:', len(stats8), 'variables')
print('L6 stats computed:', len(stats6), 'variables')

In [ ]:
# Cell 5
# Build combined cross-tabulation: L8 vs L6 side by side

combined = pd.DataFrame({
    'pct_null_L8':    stats8['pct_null'],
    'pct_null_L6':    stats6['pct_null'],
    'null_delta':     (stats6['pct_null'] - stats8['pct_null']).round(2),
    'pct_zero_L8':    stats8['pct_zero'],
    'pct_zero_L6':    stats6['pct_zero'],
    'zero_delta':     (stats6['pct_zero'] - stats8['pct_zero']).round(2),
    'pct_ceil_L8':    stats8['pct_ceiling'],
    'pct_ceil_L6':    stats6['pct_ceiling'],
})

# Sort by absolute null delta — largest scale-sensitivity at top
combined_sorted = combined.reindex(
    combined['null_delta'].abs().sort_values(ascending=False).index
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
combined_sorted

In [ ]:
# Cell 6
# Save cross-tabulation to CSV
import os
os.makedirs('/Users/karlg/Documents/Repos/_cedop/output/edop/explore', exist_ok=True)
combined_sorted.to_csv(
    '/Users/karlg/Documents/Repos/_cedop/output/edop/explore/02_missing_data_crosstab.csv'
)
print('Saved: output/edop/explore/02_missing_data_crosstab.csv')

## Part 2: Visualize null rates L8 vs L6

A scatter plot where each point is one variable. X-axis = % null at L8, Y-axis = % null at L6. Points on the diagonal = no change between levels. Points above the diagonal = more null at L6 (potential scale-degradation or N-artifact). Points below = less null at L6 (unusual — flag for investigation).

The N-confound caution: small deviations from the diagonal (say, within ±2%) are likely noise given the 12× difference in sample size. Only larger deviations warrant a "genuine degradation" interpretation.

In [ ]:
# Cell 7
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: % null scatter ---
ax = axes[0]
x = combined['pct_null_L8']
y = combined['pct_null_L6']
ax.scatter(x, y, color='steelblue', alpha=0.8, s=60)

# Diagonal reference line
lim = max(x.max(), y.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, alpha=0.5, label='no change')

# Label points with notable delta
for api_key in combined.index:
    xi, yi = combined.loc[api_key, 'pct_null_L8'], combined.loc[api_key, 'pct_null_L6']
    if abs(yi - xi) > 2:
        ax.annotate(api_key, (xi, yi), fontsize=6.5, xytext=(4, 2),
                    textcoords='offset points', color='#333')

ax.set_xlabel('% null at L8', fontsize=10)
ax.set_ylabel('% null at L6', fontsize=10)
ax.set_title('Null rate: L8 vs L6\n(above diagonal = more null at L6)', fontsize=10)
ax.legend(fontsize=8)

# --- Right: % zero scatter ---
ax = axes[1]
x2 = combined['pct_zero_L8']
y2 = combined['pct_zero_L6']
ax.scatter(x2, y2, color='darkorange', alpha=0.8, s=60)

lim2 = max(x2.max(), y2.max()) * 1.05
ax.plot([0, lim2], [0, lim2], 'k--', linewidth=0.8, alpha=0.5, label='no change')

for api_key in combined.index:
    xi, yi = combined.loc[api_key, 'pct_zero_L8'], combined.loc[api_key, 'pct_zero_L6']
    if abs(yi - xi) > 3:
        ax.annotate(api_key, (xi, yi), fontsize=6.5, xytext=(4, 2),
                    textcoords='offset points', color='#333')

ax.set_xlabel('% zero at L8', fontsize=10)
ax.set_ylabel('% zero at L6', fontsize=10)
ax.set_title('Zero rate: L8 vs L6\n(above diagonal = more zeros at L6)', fontsize=10)
ax.legend(fontsize=8)

plt.suptitle('Scale sensitivity: L8 vs L6 — null and zero rates', fontsize=12)
plt.tight_layout()
fig.savefig('/Users/karlg/Documents/Repos/_cedop/output/edop/explore/02_scale_scatter.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/edop/explore/02_scale_scatter.png')

## Part 3: Distribution shift at L6

For variables that show notable null or zero changes between levels, look at the full distribution shift — not just the null/zero rate. A variable might keep the same null rate but shift its mean or shape, which is also a form of scale-sensitivity.

We compute mean and median at both levels and flag variables where the shift is large relative to the L8 standard deviation.

In [ ]:
# Cell 8
dist_rows = []
for basin_col, api_key, scale, units, pct_var, ceiling in SCALARS:
    if basin_col not in df8.columns or basin_col not in df6.columns:
        continue
    v8 = df8[basin_col].dropna() * scale
    v6 = df6[basin_col].dropna() * scale
    if len(v8) == 0 or len(v6) == 0:
        continue
    mean_shift = v6.mean() - v8.mean()
    std8 = v8.std()
    # Standardised shift: how many L8 standard deviations does the mean move?
    std_shift = mean_shift / std8 if std8 > 0 else None
    dist_rows.append(dict(
        api_key    = api_key,
        units      = units,
        mean_L8    = round(v8.mean(), 3),
        median_L8  = round(v8.median(), 3),
        mean_L6    = round(v6.mean(), 3),
        median_L6  = round(v6.median(), 3),
        mean_shift = round(mean_shift, 3),
        std_shift  = round(std_shift, 3) if std_shift is not None else None,
    ))

dist_df = pd.DataFrame(dist_rows).set_index('api_key')
dist_df_sorted = dist_df.reindex(
    dist_df['std_shift'].abs().sort_values(ascending=False).index
)
dist_df_sorted

In [ ]:
# Cell 9
dist_df_sorted.to_csv(
    '/Users/karlg/Documents/Repos/_cedop/output/edop/explore/02_distribution_shift.csv'
)
print('Saved: output/edop/explore/02_distribution_shift.csv')

## Part 4: Scale-sensitivity classification

Combine null delta and distribution shift into a simple classification for each variable:
- **stable**: null delta < 2% AND std_shift < 0.1 — essentially unchanged between levels
- **N-artifact**: small delta, likely within noise of 12× sample size difference
- **scale-sensitive**: null delta ≥ 5% OR std_shift ≥ 0.2 — genuine change between levels
- **investigate**: unusual patterns (null rate *decreasing* at L6, or very large shifts)

These thresholds are heuristic — override by hand after reviewing the scatter plots.

In [ ]:
# Cell 10
def classify_scale_sensitivity(null_delta, std_shift):
    if null_delta is None or std_shift is None:
        return 'unknown'
    if null_delta < -2:
        return 'investigate (null decreases at L6)'
    if abs(null_delta) >= 5 or abs(std_shift) >= 0.2:
        return 'scale-sensitive'
    if abs(null_delta) < 2 and abs(std_shift) < 0.1:
        return 'stable'
    return 'N-artifact (minor delta)'

sensitivity = combined_sorted[['null_delta']].copy()
sensitivity['std_shift'] = dist_df['std_shift']
sensitivity['classification'] = sensitivity.apply(
    lambda r: classify_scale_sensitivity(r['null_delta'], r['std_shift']), axis=1
)

sensitivity_sorted = sensitivity.sort_values('classification')
print(sensitivity_sorted['classification'].value_counts())
print()
sensitivity_sorted

In [ ]:
# Cell 11
# Manual overrides — fill in after reviewing outputs
# keyed by api_key, value is the corrected classification
overrides = {}

if overrides:
    sensitivity['classification'] = sensitivity.apply(
        lambda r: overrides.get(r.name, r['classification']), axis=1
    )
    print('Overrides applied:', overrides)
else:
    print('No overrides set.')

sensitivity.to_csv(
    '/Users/karlg/Documents/Repos/_cedop/output/edop/explore/02_scale_sensitivity.csv'
)
print('Saved: output/edop/explore/02_scale_sensitivity.csv')

---
## Findings

Record observations here after reviewing the outputs above. Transfer substantive findings to `logs/exploration_log.md`.

Template:
```
**Variable / group**: xxx
**Finding**: xxx
**Implication**: xxx
```

*(add findings here)*